In [1]:
# ===== 全局运行配置 =====
MODE = "demo"          # "eval" 或 "demo"
DATASET = "cnc"        # "cnc" 或 "li"（eval 模式 + demo sample 模式使用）
PROMPT_NAME = "v1"     # prompts 目录下的 prompt 名称，不含 .txt

# demo 模式专用
DEMO_INPUT = "sample"     # "manual"（手动输入单条）或 "sample"（数据集抽样）
DEMO_TEXT = "Heavy rain caused widespread flooding in the region."  # manual 时使用
DEMO_SAMPLE_N = 3        # sample 时使用：取前 N 条

# RAG 开关与模式
USE_RAG = True
RAG_MODE = "knn_pattern"   # "pattern"、"knn" 或 "knn_pattern"
RAG_TOP_K = 5 #每种 retriever 各取 K 条 ,也就是说混合模式为10条

# LLM 参数
TEMPERATURE = 0.0      # eval 用 0 保证可复现；demo 可以调高看多样性
MAX_TOKENS = 2048


In [2]:
# ===== SPEC_02 验收 =====
# Cell A：导入、环境检查与初始化
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Python executable: {sys.executable}")
if "master_thesis" not in sys.executable.lower():
    raise RuntimeError(
        "当前 Notebook kernel 不是 Master_thesis。请在 Kernel -> Change Kernel 中选择 Master_thesis，"
        "然后 Restart Kernel 后从第一个 cell 重新运行。"
    )

try:
    import openai
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "当前 kernel 缺少 openai，说明 notebook 没有运行在 Master_thesis 环境里，"
        "或 kernel 需要重启。请切换到 Master_thesis 并 Restart Kernel。"
    ) from exc

from src.llm_client import LLMClient
client = LLMClient(temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
print(f"Base URL: {client.base_url}")
print(f"Model: {client.model}")


Python executable: D:\Anaconda3\envs\Master_thesis\python.exe
Base URL: http://127.0.0.1:1234/v1
Model: qwen/qwen3-14b


In [3]:
# Cell B：连通性测试
assert client.ping(), "LM Studio server 未启动或无法访问"
print("✓ LM Studio 连接正常")


✓ LM Studio 连接正常


In [ ]:
# Cell C：最简单的 hello 测试
resp = client.chat([{"role": "user", "content": "Reply with exactly the word 'hello' and nothing else."}])
print(f"模型回复: {resp}")


In [ ]:
# Cell D：System message 测试
resp = client.chat([
    {"role": "system", "content": "You are a translator. Translate user input to French."},
    {"role": "user", "content": "Good morning"}
])
print(f"翻译结果: {resp}")


In [ ]:
# Cell E：真实因果抽取试探（只是测 client，不评估输出质量）
resp = client.chat([
    {"role": "user", "content": "Extract cause and effect from this sentence: 'Smoking causes lung cancer.' Output in JSON format."}
])
print(f"因果输出: {resp}")


In [4]:
# ===== SPEC_03 验收 =====
# Cell F：验证 BGE embedding cache 文件存在
from pathlib import Path

bge_metadata_path = PROJECT_ROOT / "RAG Database" / "bge-small-en-v1.5_examples.jsonl"
bge_embeddings_path = PROJECT_ROOT / "RAG Database" / "bge-small-en-v1.5_embeddings.npy"
assert bge_metadata_path.exists(), f"BGE metadata cache 不存在: {bge_metadata_path}"
assert bge_embeddings_path.exists(), f"BGE embeddings cache 不存在: {bge_embeddings_path}"
print(f"OK BGE metadata cache 已就绪: {bge_metadata_path}")
print(f"OK BGE embeddings cache 已就绪: {bge_embeddings_path}")
print(f"RAG mode: {RAG_MODE} | USE_RAG={USE_RAG} | top_k={RAG_TOP_K}")


OK BGE metadata cache 已就绪: D:\Master thesis\RAG Database\bge-small-en-v1.5_examples.jsonl
OK BGE embeddings cache 已就绪: D:\Master thesis\RAG Database\bge-small-en-v1.5_embeddings.npy
RAG mode: knn_pattern | USE_RAG=True | top_k=5


In [8]:
# Cell G：选择 SPEC_03 输入并检验 Prompt 模板渲染（不调 LLM）
import importlib

import src.prompt_builder as prompt_builder
from src.data_io import load_dataset

prompt_builder = importlib.reload(prompt_builder)
build_messages = prompt_builder.build_messages

if DEMO_INPUT == "sample":
    rag_samples = load_dataset(DATASET, n=DEMO_SAMPLE_N)
    print(f"SPEC_03 sample 预览：{len(rag_samples)} 条（prompt 渲染使用第 1 条）")
    for sample in rag_samples:
        print(f"- id={sample['id']}: {sample['text']}")
    RAG_INPUT_TEXT = rag_samples[0]["text"]
    RAG_INPUT_ID = rag_samples[0]["id"]
else:
    RAG_INPUT_TEXT = DEMO_TEXT
    RAG_INPUT_ID = None

print(f"\nSPEC_03 prompt 输入 id={RAG_INPUT_ID}: {RAG_INPUT_TEXT}")
messages = build_messages(RAG_INPUT_TEXT, use_rag=False, retriever=None, top_k=0, prompt_name=PROMPT_NAME)
for m in messages:
    print(f"--- {m['role']} ---")
    print(m['content'])
    print()


SPEC_03 sample 预览：3 条（prompt 渲染使用第 1 条）
- id=1: The State alleged they hacked Sabata Petros Chale , 39 , to death in Marikana West , on December 8 , 2016 , allegedly over the allocation of low cost ( RDP ) houses at Marikana West Extension 2 .
- id=2: Chale was allegedly chased by a group of about 30 people and was hacked to death with pangas , axes and spears .
- id=3: The farmworkers ' strike resumed on Tuesday when their demands were not met .

SPEC_03 prompt 输入 id=1: The State alleged they hacked Sabata Petros Chale , 39 , to death in Marikana West , on December 8 , 2016 , allegedly over the allocation of low cost ( RDP ) houses at Marikana West Extension 2 .
--- system ---
You are a causality extraction system.

Task:
Extract causal relations from the input text and output strict JSON only.

Output schema:
{
  "has_causal": true,
  "triples": [
    {
      "cause": {"span": "..."},
      "relation": "caused",
      "effect": {"span": "..."}
    }
  ]
}

Rules:
- `has_causal` is re

In [9]:
# Cell H：按 RAG_MODE 渲染 prompt
import importlib

import src.prompt_builder as prompt_builder
import src.retriever as retriever_module

prompt_builder = importlib.reload(prompt_builder)
retriever_module = importlib.reload(retriever_module)
build_messages = prompt_builder.build_messages
create_retriever = retriever_module.create_retriever

retriever = None
if USE_RAG:
    retriever = create_retriever(
        RAG_MODE,
        metadata_path=bge_metadata_path,
        embeddings_path=bge_embeddings_path,
    )

messages_rag = build_messages(
    RAG_INPUT_TEXT,
    use_rag=USE_RAG,
    retriever=retriever,
    top_k=RAG_TOP_K,
    rag_mode=RAG_MODE,
    prompt_name=PROMPT_NAME,
)
for m in messages_rag:
    print(f"--- {m['role']} ---")
    print(m['content'])
    print()


D:\Anaconda3\envs\Master_thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6813.94it/s]


--- system ---
You are a causality extraction system.

Task:
Extract causal relations from the input text and output strict JSON only.

Output schema:
{
  "has_causal": true,
  "triples": [
    {
      "cause": {"span": "..."},
      "relation": "caused",
      "effect": {"span": "..."}
    }
  ]
}

Rules:
- `has_causal` is required and must be a boolean.
- `triples` is required and must be an array.
- If there is no causal relation, output `"has_causal": false` and `"triples": []`.
- The relation value must always be exactly `"caused"`.
- `cause.span` and `effect.span` must be original substrings from the input text.
- Do not rewrite, summarize, normalize, or translate spans.
- If a cause or effect span is a clause or event, include its subject and other necessary arguments when they are present in the input text.
- When selecting spans, preserve relevant modifiers and attributes that belong to the cause or effect, including time, location, dates, quantities, numbers, ages, and named 

In [10]:
# Cell I：观察当前 RAG_MODE 检索到的 examples
if retriever is None:
    print("RAG 未启用；如需测试检索，请设置 USE_RAG=True，并选择 RAG_MODE。")
else:
    examples = retriever.retrieve(RAG_INPUT_TEXT, top_k=RAG_TOP_K)
    print(f"检索到 {len(examples)} 个 examples:")
    for i, ex in enumerate(examples, 1):
        print(f"\n--- Example {i} ---")
        print(ex)


检索到 10 个 examples:

--- Example 1 ---
{'sentence': 'OBJECTIVE: We report a patient who developed neutropenia on <cause>clozapine</cause>, but behind the cell count decrease showed to be a <effect>diurnal variation of the white blood cells</effect> (WBC).', 'cause': 'clozapine', 'effect': 'diurnal variation of the white blood cells', 'causality_phrase': 'on', 'score': 101.3793, 'source': 'pattern'}

--- Example 2 ---
{'sentence': 'This is the second report of <effect>lactic acidosis</effect> in a patient on stavudine and <cause>lamivudine</cause>.', 'cause': 'lamivudine', 'effect': 'lactic acidosis', 'causality_phrase': 'on', 'score': 101.3793, 'source': 'pattern'}

--- Example 3 ---
{'sentence': 'CONCLUSIONS: The use of fluorouracil treatment with careful monitoring can be considered in a patient with mild <effect>allergic reactions</effect> to <cause>capecitabine</cause>.', 'cause': 'capecitabine', 'effect': 'allergic reactions', 'causality_phrase': 'to', 'score': 101.3793, 'source': 

In [11]:
# Cell J：用 SPEC_02 的 client 真正调一次 LLM（只看输出，不解析）
resp = client.chat(messages_rag if USE_RAG else messages)
print("=== LLM 原始输出 ===")
print(resp)


=== LLM 原始输出 ===
<think>
Okay, let's see. The user provided a sentence and wants me to extract causal relations from it. The task is to output JSON with "has_causal" as true or false and an array of triples if there are any.

First, I need to check the input text for any explicit or implicit causal relationships. The sentence says: "The State alleged they hacked Sabata Petros Chale, 39, to death in Marikana West, on December 8, 2016, allegedly over the allocation of low cost (RDP) houses at Marikana West Extension 2."

Looking for causal cues like "because," "due to," "as a result," etc. The phrase here is "allegedly over the allocation..." which might indicate a reason. But does that imply causation? The word "over" here probably means "because of" or "regarding." So, the allocation of houses could be the cause, and the hacking leading to death is the effect.

But wait, the structure is "allegedly over the allocation..." which might not be a direct causal link. It's more like stating 

In [ ]:
# ===== SPEC_04 验收 =====
# Cell K：parse_output 鲁棒性快速测试（不调 LLM，用构造字符串）
import json
from src.generator import generate, parse_output
from src.data_io import load_dataset

test_cases = [
    '{"has_causal": true, "triples": []}',
    'Here is the result:\n{"has_causal": false, "triples": []}',
    '```json\n{"has_causal": true, "triples": []}\n```',
    'Sure!\n```\n{"has_causal": false, "triples": []}\n```\nDone.',
    '<think>reasoning</think>\n{"has_causal": true, "triples": []}',
]
for i, tc in enumerate(test_cases, 1):
    try:
        result = parse_output(tc)
        print(f"OK 用例 {i}: {result}")
    except Exception as e:
        print(f"FAIL 用例 {i} 解析失败: {e}")


In [ ]:
# Cell L：单条 demo（manual 模式）
if MODE == "demo" and DEMO_INPUT == "manual":
    result = generate(
        text=DEMO_TEXT,
        sample_id=None,
        client=client,
        retriever=retriever if USE_RAG else None,
        use_rag=USE_RAG,
        top_k=RAG_TOP_K,
        rag_mode=RAG_MODE,
        prompt_name=PROMPT_NAME,
    )
    print("=== Demo 单条输出 ===")
    print(json.dumps(result, indent=2, ensure_ascii=False))


In [6]:
# Cell M：数据集抽样 demo（sample 模式）
if MODE == "demo" and DEMO_INPUT == "sample":
    samples = load_dataset(DATASET, n=DEMO_SAMPLE_N)
    for s in samples:
        result = generate(
            text=s["text"],
            sample_id=s["id"],
            client=client,
            retriever=retriever if USE_RAG else None,
            use_rag=USE_RAG,
            top_k=RAG_TOP_K,
            rag_mode=RAG_MODE,
            prompt_name=PROMPT_NAME,
        )
        print(f"\n--- id={s['id']} ---")
        print(f"输入: {s['text']}")
        print("Gold:")
        print(json.dumps({"has_causal": s["has_causal"], "triples": s["relations"]}, indent=2, ensure_ascii=False))
        print("Pred:")
        print(json.dumps(result, indent=2, ensure_ascii=False))


NameError: name 'generate' is not defined